In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pycaret.classification import *
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("C:\\Users\\ripa_\\Desktop\\Programing\\IndyCar_Project\\LigaMX\\datasets\\LigaMX_dataset_v3.csv")

In [3]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Date", "Equipo"])

In [4]:
df.head()

,Date,Time,Round,Day,Venue,Opponent,Referee,Equipo,Torneo,Temporada,...,TeamAwayForm5,OpponentForm5,OpponentWinRate5,OpponentSeasonPts,OpponentHomeForm5,OpponentAwayForm5,H2HWinRate,H2HLast5,FormDiff,SeasonPtsDiff
0,2014-07-18,19:30,Jornada 1,Fri,Away,Tijuana,Luis Enrique Santander,Puebla,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0
1,2014-07-18,19:30,Jornada 1,Fri,Away,Queretaro,Erick Yair Miranda,Pumas UNAM,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0
2,2014-07-18,19:30,Jornada 1,Fri,Home,Pumas UNAM,Erick Yair Miranda,Queretaro,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0
3,2014-07-18,19:30,Jornada 1,Fri,Home,Puebla,Luis Enrique Santander,Tijuana,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0
4,2014-07-19,21:00,Jornada 1,Sat,Home,Tigres UANL,Erim Ramírez,Atlas,Apertura,2014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
drop_cols = [
    "Date", "Time", "Round", "Day", "Venue", "Opponent", "Referee", "Equipo", "Torneo", "Temporada", "Result", "Points"
]

cutoff = df["Date"].quantile(0.95)
data = df[df["Date"] < cutoff].drop(columns=drop_cols)
data_unseen = df[df["Date"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["ResultID"].sort_values())

OponentElo          -0.110425
OpponentAwayForm5   -0.028026
OpponentSeasonPts   -0.024115
DayID               -0.024010
OpponentForm5       -0.017161
OpponentWinRate5    -0.015994
TemporadaID         -0.011490
MatchDay            -0.010516
TorneoID            -0.010137
RoundID             -0.008906
TeamID              -0.002708
OpponentHomeForm5    0.003189
RefereeID            0.008271
TimeID               0.013922
TeamSeasonPts        0.022017
TeamHomeForm5        0.026920
OpponentID           0.028179
TeamWinRate5         0.050780
H2HWinRate           0.054856
TeamAwayForm5        0.055212
FormDiff             0.058839
TeamForm5            0.059213
SeasonPtsDiff        0.073546
VenueID              0.109879
H2HLast5             0.124469
TeamElo              0.143408
EloDiff              0.182490
ResultID             1.000000
Name: ResultID, dtype: float64


In [6]:
df = df.drop(columns=drop_cols)

In [7]:
print(df.columns.tolist())

['VenueID', 'OpponentID', 'TeamID', 'RefereeID', 'RoundID', 'TemporadaID', 'TorneoID', 'TimeID', 'DayID', 'ResultID', 'TeamElo', 'OponentElo', 'EloDiff', 'TeamForm5', 'TeamWinRate5', 'TeamSeasonPts', 'MatchDay', 'TeamHomeForm5', 'TeamAwayForm5', 'OpponentForm5', 'OpponentWinRate5', 'OpponentSeasonPts', 'OpponentHomeForm5', 'OpponentAwayForm5', 'H2HWinRate', 'H2HLast5', 'FormDiff', 'SeasonPtsDiff']


In [8]:
exp = setup(
    data=data, 
    target="ResultID", 
    session_id=123, 
    fold_strategy="timeseries",
    data_split_shuffle=False,
    data_split_stratify=False,
    fold_shuffle=False
)

,Description,Value
0,Session id,123
1,Target,ResultID
2,Target type,Multiclass
3,Original data shape,"(7032, 28)"
4,Transformed data shape,"(7032, 28)"
5,Transformed train set shape,"(4922, 28)"
6,Transformed test set shape,"(2110, 28)"
7,Numeric features,27
8,Rows with missing values,5.5%
9,Preprocess,True


In [9]:
compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
et,Extra Trees Classifier,0.5045,0.6651,0.5045,0.4945,0.4798,0.2353,0.2435,0.1020
ridge,Ridge Classifier,0.5013,0.0000,0.5013,0.4852,0.4397,0.2232,0.2429,0.0110
rf,Random Forest Classifier,0.4984,0.6641,0.4984,0.4833,0.4709,0.2246,0.2326,0.1360
lr,Logistic Regression,0.4975,0.0000,0.4975,0.4688,0.4552,0.2191,0.2347,1.0140
lda,Linear Discriminant Analysis,0.4969,0.0000,0.4969,0.4713,0.4458,0.2202,0.2365,0.0100
gbc,Gradient Boosting Classifier,0.4940,0.0000,0.4940,0.4827,0.4720,0.2205,0.2271,0.4990
catboost,CatBoost Classifier,0.4937,0.6696,0.4937,0.4819,0.4782,0.2230,0.2272,2.1320
lightgbm,Light Gradient Boosting Machine,0.4855,0.6657,0.4855,0.4709,0.4688,0.2091,0.2132,0.3760
xgboost,Extreme Gradient Boosting,0.4843,0.6581,0.4843,0.4731,0.4708,0.2090,0.2124,0.2480
ada,Ada Boost Classifier,0.4801,0.0000,0.4801,0.4723,0.4572,0.2015,0.2096,0.0630


ExtraTreesClassifier(bootstrap=False, ccp_alpha=0.0, class_weight=None,
                     criterion='gini', max_depth=None, max_features='sqrt',
                     max_leaf_nodes=None, max_samples=None,
                     min_impurity_decrease=0.0, min_samples_leaf=1,
                     min_samples_split=2, min_weight_fraction_leaf=0.0,
                     monotonic_cst=None, n_estimators=100, n_jobs=-1,
                     oob_score=False, random_state=123, verbose=0,
                     warm_start=False)

In [10]:
cat = create_model('catboost')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4653,0.6232,0.4653,0.4726,0.4623,0.1740,0.1767
1,0.4564,0.6393,0.4564,0.4436,0.4236,0.1781,0.1886
2,0.4855,0.6588,0.4855,0.4900,0.4875,0.2259,0.2260
3,0.4765,0.6514,0.4765,0.4688,0.4681,0.2025,0.2045
4,0.4855,0.6867,0.4855,0.4772,0.4805,0.2175,0.2179
5,0.4765,0.6597,0.4765,0.4627,0.4677,0.1982,0.1991
6,0.5235,0.6912,0.5235,0.4908,0.4986,0.2545,0.2593
7,0.5526,0.7215,0.5526,0.5294,0.5337,0.3005,0.3046
8,0.5481,0.7159,0.5481,0.5318,0.5228,0.2864,0.2942


In [11]:
cat_tune = tune_model(cat)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4676,0.6339,0.4676,0.4724,0.4574,0.1756,0.1808
1,0.4452,0.6677,0.4452,0.4307,0.3835,0.1589,0.1773
2,0.4899,0.6706,0.4899,0.4831,0.4860,0.2280,0.2282
3,0.4810,0.6686,0.4810,0.4757,0.4774,0.2131,0.2135
4,0.5190,0.6935,0.5190,0.4965,0.5000,0.2611,0.2655
5,0.5347,0.6744,0.5347,0.5179,0.5124,0.2785,0.2852
6,0.5526,0.7174,0.5526,0.5073,0.5132,0.2942,0.3045
7,0.5548,0.7399,0.5548,0.5277,0.5220,0.2961,0.3059
8,0.5570,0.7098,0.5570,0.5280,0.5182,0.2959,0.3077


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [23]:
predict_model(cat_tune);
#predict_model(cat);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,CatBoost Classifier,0.5246,0.6967,0.5246,0.4870,0.4903,0.2592,0.2680


In [25]:
newpred = predict_model(cat_tune, data=data_unseen)
#newpred = predict_model(cat, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,CatBoost Classifier,0.5914,0.7231,0.5914,0.5693,0.5742,0.3471,0.3519


In [12]:
rf = create_model('rf')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4564,0.6349,0.4564,0.4538,0.4382,0.1509,0.1576
1,0.4676,0.6315,0.4676,0.4862,0.4238,0.1937,0.2117
2,0.4966,0.6439,0.4966,0.4735,0.4815,0.2334,0.2355
3,0.4653,0.6471,0.4653,0.4500,0.4506,0.1826,0.1856
4,0.4966,0.6720,0.4966,0.4739,0.4750,0.2247,0.2295
5,0.5168,0.6682,0.5168,0.4891,0.4913,0.2509,0.2570
6,0.5459,0.7014,0.5459,0.5233,0.5258,0.2908,0.2956
7,0.5302,0.6981,0.5302,0.5021,0.5032,0.2601,0.2666
8,0.5369,0.7051,0.5369,0.5106,0.4955,0.2621,0.2742


In [13]:
rf_tune = tune_model(rf)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4609,0.6379,0.4609,0.4900,0.4409,0.1726,0.1848
1,0.4452,0.6531,0.4452,0.4337,0.4014,0.1606,0.1784
2,0.4989,0.6652,0.4989,0.4987,0.4983,0.2435,0.2438
3,0.4922,0.6545,0.4922,0.5026,0.4921,0.2358,0.2381
4,0.4899,0.6845,0.4899,0.4792,0.4825,0.2230,0.2241
5,0.4877,0.6651,0.4877,0.4698,0.4744,0.2125,0.2148
6,0.5056,0.6954,0.5056,0.4831,0.4906,0.2329,0.2350
7,0.5213,0.7027,0.5213,0.5105,0.5147,0.2611,0.2618
8,0.5481,0.6907,0.5481,0.5389,0.5382,0.2965,0.2993


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [27]:
predict_model(rf_tune);
#predict_model(rf);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.5076,0.6759,0.5076,0.4641,0.4705,0.2318,0.2402


In [28]:
newpred = predict_model(rf_tune, data=data_unseen)
#newpred = predict_model(rf, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.5726,0.6948,0.5726,0.5519,0.5546,0.3161,0.3213


In [14]:
lgbm = create_model('lightgbm')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4430,0.6045,0.4430,0.4482,0.4410,0.1347,0.1361
1,0.4430,0.6207,0.4430,0.4241,0.4149,0.1586,0.1663
2,0.4743,0.6584,0.4743,0.4756,0.4749,0.2077,0.2077
3,0.4519,0.6359,0.4519,0.4472,0.4491,0.1695,0.1697
4,0.4765,0.6804,0.4765,0.4622,0.4670,0.2010,0.2021
5,0.4855,0.6662,0.4855,0.4587,0.4648,0.2049,0.2084
6,0.5302,0.6975,0.5302,0.4985,0.5084,0.2677,0.2715
7,0.5302,0.7154,0.5302,0.5085,0.5074,0.2616,0.2673
8,0.5638,0.7222,0.5638,0.5451,0.5351,0.3101,0.3193


In [15]:
lgbm_tune = tune_model(lgbm)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4765,0.6325,0.4765,0.4936,0.4556,0.1827,0.1957
1,0.4452,0.6602,0.4452,0.3962,0.3684,0.1582,0.1789
2,0.5101,0.6703,0.5101,0.4909,0.4910,0.2506,0.2554
3,0.4855,0.6655,0.4855,0.4864,0.4634,0.2067,0.2142
4,0.5123,0.6964,0.5123,0.4721,0.4680,0.2420,0.2542
5,0.5168,0.6748,0.5168,0.4999,0.4641,0.2404,0.2569
6,0.5548,0.7029,0.5548,0.5312,0.5032,0.2914,0.3084
7,0.5593,0.7230,0.5593,0.5780,0.5184,0.2971,0.3129
8,0.5727,0.7250,0.5727,0.5867,0.5165,0.3143,0.3348


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [30]:
predict_model(lgbm_tune);
#predict_model(lgbm);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.5332,0.6903,0.5332,0.5108,0.4623,0.2616,0.2845


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [32]:
newpred = predict_model(lgbm_tune, data=data_unseen)
#newpred = predict_model(lgbm, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.5860,0.7226,0.5860,0.5242,0.5095,0.3131,0.3446


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [16]:
et = create_model('et')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4832,0.6389,0.4832,0.4891,0.4787,0.1996,0.2029
1,0.4497,0.6314,0.4497,0.4940,0.4037,0.1661,0.1902
2,0.4698,0.6453,0.4698,0.4662,0.4677,0.1986,0.1988
3,0.4832,0.6277,0.4832,0.4717,0.4648,0.2068,0.2120
4,0.5101,0.6723,0.5101,0.4957,0.4981,0.2498,0.2522
5,0.5123,0.6646,0.5123,0.4848,0.4828,0.2418,0.2494
6,0.5459,0.7001,0.5459,0.5175,0.5223,0.2899,0.2953
7,0.5772,0.7201,0.5772,0.5600,0.5515,0.3338,0.3423
8,0.5391,0.7073,0.5391,0.5044,0.4958,0.2657,0.2778


In [17]:
et_tune = tune_model(et)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4810,0.6472,0.4810,0.4865,0.4758,0.1972,0.2007
1,0.4452,0.6456,0.4452,0.4632,0.3743,0.1582,0.1812
2,0.4877,0.6516,0.4877,0.4658,0.4698,0.2173,0.2207
3,0.4877,0.6653,0.4877,0.5233,0.4423,0.2035,0.2190
4,0.5101,0.6939,0.5101,0.3817,0.4358,0.2330,0.2538
5,0.5034,0.6700,0.5034,0.5025,0.4372,0.2158,0.2350
6,0.5570,0.7043,0.5570,0.5436,0.4895,0.2910,0.3136
7,0.5548,0.7171,0.5548,0.6580,0.4961,0.2850,0.3063
8,0.5638,0.7136,0.5638,0.5492,0.4977,0.2982,0.3201


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [34]:
#predict_model(et_tune);
predict_model(et);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,0.5204,0.6737,0.5204,0.4921,0.4840,0.2504,0.2608


In [35]:
#newpred = predict_model(et_tune, data=data_unseen)
newpred = predict_model(et, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,0.5403,0.6967,0.5403,0.5051,0.5174,0.2673,0.2711


In [18]:
xgb = create_model('xgboost')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4519,0.6122,0.4519,0.4586,0.4506,0.1509,0.1525
1,0.4497,0.6265,0.4497,0.4422,0.4298,0.1694,0.1764
2,0.4787,0.6539,0.4787,0.4810,0.4798,0.2146,0.2147
3,0.4541,0.6312,0.4541,0.4475,0.4502,0.1723,0.1725
4,0.4877,0.6638,0.4877,0.4797,0.4823,0.2205,0.2212
5,0.4609,0.6441,0.4609,0.4413,0.4455,0.1688,0.1710
6,0.5056,0.6709,0.5056,0.4771,0.4868,0.2306,0.2333
7,0.5235,0.7117,0.5235,0.5017,0.5062,0.2557,0.2591
8,0.5660,0.7125,0.5660,0.5520,0.5474,0.3184,0.3241


In [19]:
xgb_tune = tune_model(xgb)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4944,0.6501,0.4944,0.5241,0.4283,0.1572,0.1911
1,0.4228,0.6581,0.4228,0.3114,0.3330,0.1247,0.1649
2,0.4944,0.6622,0.4944,0.3626,0.4115,0.2132,0.2457
3,0.4743,0.6649,0.4743,0.3471,0.3962,0.1777,0.2024
4,0.5168,0.6808,0.5168,0.3842,0.4342,0.2411,0.2750
5,0.5280,0.6639,0.5280,0.3901,0.4477,0.2527,0.2794
6,0.5660,0.7057,0.5660,0.4307,0.4859,0.3040,0.3352
7,0.5414,0.7253,0.5414,0.4129,0.4672,0.2611,0.2851
8,0.5459,0.7238,0.5459,0.4176,0.4713,0.2671,0.2920


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [37]:
#predict_model(xgb_tune);
predict_model(xgb);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extreme Gradient Boosting,0.4896,0.6696,0.4896,0.4716,0.4769,0.2156,0.2176


In [39]:
#newpred = predict_model(xgb_tune, data=data_unseen)
newpred = predict_model(xgb, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extreme Gradient Boosting,0.5376,0.6934,0.5376,0.5344,0.5354,0.2774,0.2777


In [20]:
nb = create_model('nb')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4720,0.6346,0.4720,0.4891,0.4625,0.1898,0.1970
1,0.4362,0.6297,0.4362,0.3949,0.3749,0.1457,0.1645
2,0.4653,0.6481,0.4653,0.4722,0.4649,0.1950,0.1965
3,0.4564,0.6342,0.4564,0.4657,0.4558,0.1810,0.1830
4,0.4787,0.6651,0.4787,0.4707,0.4720,0.2069,0.2082
5,0.4810,0.6633,0.4810,0.4634,0.4662,0.2005,0.2032
6,0.5011,0.6839,0.5011,0.4740,0.4838,0.2252,0.2275
7,0.5011,0.6774,0.5011,0.4680,0.4783,0.2188,0.2223
8,0.5145,0.6823,0.5145,0.4863,0.4908,0.2364,0.2412


In [21]:
nb_tune = tune_model(nb)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4497,0.5854,0.4497,0.4354,0.4410,0.1256,0.1261
1,0.4273,0.6188,0.4273,0.2918,0.3463,0.1308,0.1497
2,0.4810,0.6598,0.4810,0.3443,0.4013,0.1916,0.2137
3,0.4631,0.6239,0.4631,0.3335,0.3876,0.1607,0.1788
4,0.4743,0.6489,0.4743,0.3448,0.3993,0.1740,0.1926
5,0.5190,0.6547,0.5190,0.3821,0.4399,0.2387,0.2631
6,0.5101,0.6474,0.5101,0.3844,0.4384,0.2135,0.2325
7,0.5056,0.6560,0.5056,0.3834,0.4360,0.2036,0.2211
8,0.5078,0.6481,0.5078,0.3863,0.4388,0.2058,0.2231


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [41]:
#predict_model(nb_tune);
predict_model(nb);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Naive Bayes,0.5066,0.6662,0.5066,0.4734,0.4770,0.2326,0.2392


In [43]:
#newpred = predict_model(nb_tune, data=data_unseen)
newpred = predict_model(nb, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Naive Bayes,0.5511,0.6854,0.5511,0.5285,0.5332,0.2807,0.2853


In [44]:
blend1 = blend_models([cat_tune, lgbm_tune])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4720,0.6387,0.4720,0.4803,0.4547,0.1812,0.1903
1,0.4564,0.6695,0.4564,0.4064,0.3842,0.1756,0.1971
2,0.5145,0.6729,0.5145,0.4993,0.5026,0.2606,0.2629
3,0.5011,0.6702,0.5011,0.4920,0.4921,0.2390,0.2411
4,0.4944,0.6975,0.4944,0.4538,0.4625,0.2190,0.2255
5,0.5235,0.6761,0.5235,0.4967,0.4856,0.2559,0.2671
6,0.5570,0.7144,0.5570,0.5116,0.5075,0.2970,0.3118
7,0.5548,0.7353,0.5548,0.5571,0.5230,0.2934,0.3054
8,0.5705,0.7184,0.5705,0.5741,0.5226,0.3128,0.3304


In [45]:
blend1_tune = tune_model(blend1)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4720,0.6381,0.4720,0.4799,0.4556,0.1806,0.1893
1,0.4586,0.6697,0.4586,0.4227,0.3887,0.1790,0.2008
2,0.5101,0.6728,0.5101,0.4962,0.4997,0.2546,0.2564
3,0.5056,0.6704,0.5056,0.4972,0.4968,0.2458,0.2479
4,0.4877,0.6973,0.4877,0.4472,0.4576,0.2096,0.2151
5,0.5302,0.6760,0.5302,0.5119,0.4966,0.2671,0.2779
6,0.5570,0.7148,0.5570,0.5174,0.5096,0.2973,0.3118
7,0.5548,0.7359,0.5548,0.5506,0.5234,0.2941,0.3054
8,0.5660,0.7179,0.5660,0.5638,0.5189,0.3060,0.3229


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [61]:
predict_model(blend1_tune);
#predict_model(blend1);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5294,0.6970,0.5294,0.4838,0.4746,0.2598,0.2762


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [92]:
newpred = predict_model(blend1_tune, data=data_unseen)
#newpred = predict_model(blend1, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5968,0.7263,0.5968,0.5604,0.5564,0.3420,0.3583


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [46]:
blend2 = blend_models([cat_tune, et])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4653,0.6405,0.4653,0.4753,0.4556,0.1767,0.1825
1,0.4698,0.6598,0.4698,0.5073,0.4148,0.1962,0.2226
2,0.4966,0.6647,0.4966,0.4933,0.4946,0.2392,0.2394
3,0.4810,0.6578,0.4810,0.4709,0.4720,0.2093,0.2112
4,0.5168,0.6908,0.5168,0.4938,0.4991,0.2585,0.2619
5,0.5391,0.6746,0.5391,0.5165,0.5107,0.2837,0.2922
6,0.5548,0.7156,0.5548,0.5196,0.5201,0.2986,0.3081
7,0.5749,0.7398,0.5749,0.5515,0.5432,0.3283,0.3386
8,0.5660,0.7173,0.5660,0.5659,0.5257,0.3077,0.3226


In [47]:
blend2_tune = tune_model(blend2)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4698,0.6410,0.4698,0.4802,0.4609,0.1830,0.1887
1,0.4631,0.6565,0.4631,0.5028,0.4090,0.1860,0.2115
2,0.4989,0.6623,0.4989,0.4949,0.4963,0.2420,0.2423
3,0.4877,0.6548,0.4877,0.4780,0.4778,0.2187,0.2211
4,0.5257,0.6896,0.5257,0.5018,0.5071,0.2720,0.2758
5,0.5414,0.6737,0.5414,0.5247,0.5135,0.2867,0.2959
6,0.5503,0.7141,0.5503,0.5139,0.5167,0.2921,0.3010
7,0.5861,0.7382,0.5861,0.5651,0.5546,0.3460,0.3568
8,0.5570,0.7167,0.5570,0.5587,0.5178,0.2935,0.3077


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [65]:
predict_model(blend2_tune);
#predict_model(blend2);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5294,0.6920,0.5294,0.4977,0.4876,0.2631,0.2757


In [67]:
#newpred = predict_model(blend2_tune, data=data_unseen)
newpred = predict_model(blend2, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5806,0.7175,0.5806,0.5528,0.5560,0.3249,0.3327


In [48]:
blend3 = blend_models([cat_tune, nb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4676,0.6380,0.4676,0.4849,0.4570,0.1814,0.1888
1,0.4586,0.6498,0.4586,0.4552,0.3982,0.1793,0.2024
2,0.4653,0.6621,0.4653,0.4692,0.4635,0.1938,0.1953
3,0.4810,0.6534,0.4810,0.4821,0.4791,0.2154,0.2166
4,0.4944,0.6835,0.4944,0.4775,0.4814,0.2267,0.2291
5,0.4989,0.6733,0.4989,0.4786,0.4806,0.2260,0.2299
6,0.5324,0.7065,0.5324,0.4984,0.5072,0.2687,0.2737
7,0.5145,0.7116,0.5145,0.4733,0.4822,0.2343,0.2409
8,0.5436,0.7027,0.5436,0.5266,0.5170,0.2786,0.2865


In [49]:
blend3_tune = tune_model(blend3)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4720,0.6374,0.4720,0.4846,0.4604,0.1838,0.1911
1,0.4586,0.6610,0.4586,0.4653,0.3963,0.1792,0.2026
2,0.4944,0.6680,0.4944,0.4881,0.4890,0.2334,0.2346
3,0.4765,0.6642,0.4765,0.4750,0.4736,0.2073,0.2083
4,0.5056,0.6921,0.5056,0.4855,0.4883,0.2410,0.2448
5,0.5213,0.6747,0.5213,0.5009,0.4981,0.2578,0.2640
6,0.5503,0.7160,0.5503,0.5134,0.5200,0.2942,0.3014
7,0.5459,0.7291,0.5459,0.5272,0.5165,0.2823,0.2914
8,0.5615,0.7098,0.5615,0.5366,0.5283,0.3051,0.3153


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [69]:
predict_model(blend3_tune);
#predict_model(blend3);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5232,0.6944,0.5232,0.4819,0.4836,0.2551,0.2658


In [71]:
newpred = predict_model(blend3_tune, data=data_unseen)
#newpred = predict_model(blend3, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5833,0.7205,0.5833,0.5604,0.5623,0.3303,0.3372


In [50]:
blend4 = blend_models([cat_tune, et, nb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4743,0.6422,0.4743,0.4952,0.4638,0.1917,0.2000
1,0.4720,0.6501,0.4720,0.4957,0.4163,0.1997,0.2271
2,0.4832,0.6621,0.4832,0.4810,0.4792,0.2185,0.2199
3,0.4765,0.6540,0.4765,0.4726,0.4713,0.2052,0.2067
4,0.4989,0.6842,0.4989,0.4807,0.4837,0.2316,0.2348
5,0.5213,0.6754,0.5213,0.5041,0.4963,0.2567,0.2644
6,0.5369,0.7113,0.5369,0.5041,0.5093,0.2737,0.2800
7,0.5414,0.7207,0.5414,0.5134,0.5082,0.2745,0.2839
8,0.5503,0.7103,0.5503,0.5333,0.5110,0.2838,0.2965


In [51]:
blend4_tune = tune_model(blend4)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4810,0.6413,0.4810,0.4904,0.4706,0.1964,0.2031
1,0.4676,0.6610,0.4676,0.5122,0.4113,0.1928,0.2194
2,0.4832,0.6670,0.4832,0.4776,0.4788,0.2178,0.2186
3,0.4855,0.6611,0.4855,0.4791,0.4795,0.2183,0.2197
4,0.5168,0.6911,0.5168,0.4966,0.5007,0.2589,0.2622
5,0.5369,0.6772,0.5369,0.5204,0.5112,0.2805,0.2887
6,0.5548,0.7175,0.5548,0.5244,0.5234,0.2995,0.3083
7,0.5638,0.7358,0.5638,0.5480,0.5323,0.3096,0.3204
8,0.5705,0.7162,0.5705,0.5564,0.5260,0.3145,0.3301


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [73]:
predict_model(blend4_tune);
#predict_model(blend4);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5237,0.6951,0.5237,0.4867,0.4822,0.2547,0.2665


In [75]:
newpred = predict_model(blend4_tune, data=data_unseen)
#newpred = predict_model(blend4, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5887,0.7194,0.5887,0.5618,0.5610,0.3343,0.3447


In [52]:
blend5 = blend_models([cat_tune, lgbm_tune, et])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4765,0.6428,0.4765,0.4849,0.4635,0.1885,0.1959
1,0.4631,0.6658,0.4631,0.4894,0.3992,0.1857,0.2114
2,0.5101,0.6689,0.5101,0.4953,0.4993,0.2545,0.2564
3,0.4787,0.6635,0.4787,0.4655,0.4652,0.2027,0.2057
4,0.5123,0.6946,0.5123,0.4858,0.4880,0.2483,0.2540
5,0.5369,0.6767,0.5369,0.5293,0.4995,0.2758,0.2892
6,0.5503,0.7148,0.5503,0.5071,0.5075,0.2886,0.3006
7,0.5727,0.7371,0.5727,0.5766,0.5369,0.3208,0.3350
8,0.5660,0.7219,0.5660,0.5784,0.5183,0.3053,0.3230


In [53]:
blend5_tune = tune_model(blend5)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4855,0.6439,0.4855,0.5008,0.4677,0.1993,0.2109
1,0.4564,0.6672,0.4564,0.4426,0.3829,0.1752,0.1999
2,0.5190,0.6713,0.5190,0.4984,0.5011,0.2652,0.2693
3,0.4832,0.6657,0.4832,0.4741,0.4672,0.2071,0.2115
4,0.5101,0.6975,0.5101,0.4743,0.4757,0.2416,0.2502
5,0.5347,0.6770,0.5347,0.5372,0.4885,0.2696,0.2864
6,0.5593,0.7120,0.5593,0.5303,0.5129,0.3005,0.3156
7,0.5682,0.7340,0.5682,0.5884,0.5283,0.3117,0.3279
8,0.5749,0.7241,0.5749,0.5769,0.5189,0.3182,0.3385


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [77]:
#predict_model(blend5_tune);
predict_model(blend5);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5303,0.6954,0.5303,0.4907,0.4767,0.2614,0.2778


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [79]:
#newpred = predict_model(blend5_tune, data=data_unseen)
newpred = predict_model(blend5, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5968,0.7220,0.5968,0.5648,0.5577,0.3429,0.3587


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [54]:
blend6 = blend_models([rf, et])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4631,0.6419,0.4631,0.4726,0.4536,0.1687,0.1742
1,0.4541,0.6403,0.4541,0.4691,0.4030,0.1729,0.1949
2,0.4810,0.6513,0.4810,0.4678,0.4728,0.2121,0.2129
3,0.4832,0.6430,0.4832,0.4686,0.4680,0.2092,0.2129
4,0.5190,0.6796,0.5190,0.4981,0.4992,0.2601,0.2649
5,0.5213,0.6711,0.5213,0.4909,0.4875,0.2545,0.2641
6,0.5391,0.7062,0.5391,0.5096,0.5121,0.2769,0.2835
7,0.5526,0.7162,0.5526,0.5459,0.5197,0.2902,0.3017
8,0.5570,0.7116,0.5570,0.5408,0.5121,0.2928,0.3077


In [55]:
blend6_tune = tune_model(blend6)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4653,0.6418,0.4653,0.4754,0.4555,0.1722,0.1780
1,0.4609,0.6397,0.4609,0.4800,0.4105,0.1831,0.2068
2,0.4832,0.6515,0.4832,0.4718,0.4760,0.2158,0.2166
3,0.4787,0.6406,0.4787,0.4651,0.4645,0.2028,0.2064
4,0.5213,0.6794,0.5213,0.5000,0.5032,0.2647,0.2687
5,0.5168,0.6709,0.5168,0.4851,0.4840,0.2481,0.2569
6,0.5436,0.7058,0.5436,0.5121,0.5167,0.2845,0.2909
7,0.5570,0.7178,0.5570,0.5445,0.5260,0.2987,0.3092
8,0.5503,0.7115,0.5503,0.5313,0.5065,0.2824,0.2964


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [81]:
predict_model(blend6_tune);
#predict_model(blend6);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5180,0.6808,0.5180,0.4770,0.4777,0.2465,0.2570


In [82]:
newpred = predict_model(blend6_tune, data=data_unseen)
newpred = predict_model(blend6, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5753,0.7027,0.5753,0.5515,0.5541,0.3173,0.3240


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5833,0.7028,0.5833,0.5596,0.5613,0.3300,0.3373


In [56]:
blend7 = blend_models([lgbm_tune, xgb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4676,0.6282,0.4676,0.4745,0.4644,0.1736,0.1762
1,0.4586,0.6460,0.4586,0.4510,0.4284,0.1817,0.1932
2,0.4832,0.6643,0.4832,0.4826,0.4828,0.2202,0.2203
3,0.4676,0.6457,0.4676,0.4581,0.4615,0.1911,0.1917
4,0.4743,0.6802,0.4743,0.4600,0.4647,0.1976,0.1987
5,0.4832,0.6565,0.4832,0.4612,0.4634,0.2007,0.2045
6,0.5280,0.6872,0.5280,0.4958,0.5018,0.2604,0.2659
7,0.5347,0.7212,0.5347,0.5098,0.5102,0.2683,0.2743
8,0.5570,0.7270,0.5570,0.5385,0.5272,0.2988,0.3082


In [57]:
blend7_tune = tune_model(blend7)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4810,0.6353,0.4810,0.4837,0.4660,0.1831,0.1909
1,0.4609,0.6630,0.4609,0.4484,0.4118,0.1835,0.2017
2,0.5034,0.6711,0.5034,0.4924,0.4966,0.2468,0.2475
3,0.4855,0.6575,0.4855,0.4690,0.4713,0.2133,0.2161
4,0.4922,0.6919,0.4922,0.4695,0.4736,0.2198,0.2234
5,0.5145,0.6676,0.5145,0.4894,0.4798,0.2426,0.2525
6,0.5436,0.6995,0.5436,0.5091,0.5082,0.2801,0.2900
7,0.5593,0.7262,0.5593,0.5513,0.5236,0.3002,0.3128
8,0.5705,0.7331,0.5705,0.5563,0.5237,0.3138,0.3303


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [84]:
predict_model(blend7_tune);
#predict_model(blend7);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5218,0.6911,0.5218,0.4725,0.4729,0.2500,0.2634


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [86]:
newpred = predict_model(blend7_tune, data=data_unseen)
#newpred = predict_model(blend7, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5941,0.7226,0.5941,0.5580,0.5557,0.3380,0.3533


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [58]:
blend8 = blend_models([cat_tune, rf, lgbm_tune, et, xgb, nb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4743,0.6417,0.4743,0.4856,0.4618,0.1845,0.1921
1,0.4720,0.6552,0.4720,0.4727,0.4168,0.2000,0.2250
2,0.5011,0.6670,0.5011,0.4928,0.4958,0.2442,0.2449
3,0.4698,0.6568,0.4698,0.4591,0.4611,0.1929,0.1944
4,0.5168,0.6903,0.5168,0.5003,0.5018,0.2585,0.2619
5,0.5168,0.6720,0.5168,0.4931,0.4877,0.2484,0.2563
6,0.5503,0.7110,0.5503,0.5161,0.5186,0.2928,0.3011
7,0.5436,0.7265,0.5436,0.5277,0.5107,0.2767,0.2871
8,0.5682,0.7239,0.5682,0.5505,0.5244,0.3113,0.3264


In [59]:
blend8_tune = tune_model(blend8)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4787,0.6445,0.4787,0.4895,0.4663,0.1909,0.1986
1,0.4832,0.6580,0.4832,0.4950,0.4238,0.2167,0.2443
2,0.5011,0.6682,0.5011,0.4925,0.4949,0.2434,0.2445
3,0.4899,0.6563,0.4899,0.4810,0.4799,0.2219,0.2245
4,0.5145,0.6923,0.5145,0.4908,0.4953,0.2541,0.2582
5,0.5280,0.6735,0.5280,0.5072,0.4930,0.2634,0.2743
6,0.5570,0.7111,0.5570,0.5215,0.5222,0.3020,0.3118
7,0.5660,0.7314,0.5660,0.5509,0.5303,0.3115,0.3238
8,0.5817,0.7257,0.5817,0.5854,0.5371,0.3317,0.3490


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [88]:
predict_model(blend8_tune);
#predict_model(blend8);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5261,0.6924,0.5261,0.4883,0.4772,0.2560,0.2707


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [90]:
newpred = predict_model(blend8_tune, data=data_unseen)
#newpred = predict_model(blend8, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5968,0.7187,0.5968,0.5672,0.5675,0.3472,0.3582


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [93]:
save_model(blend1_tune, "LigaMX_model_v2")

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['VenueID', 'OpponentID', 'TeamID',
                                              'RefereeID', 'RoundID',
                                              'TemporadaID', 'TorneoID',
                                              'TimeID', 'DayID', 'TeamElo',
                                              'OponentElo', 'EloDiff',
                                              'TeamForm5', 'TeamWinRate5',
                                              'TeamSeasonPts', 'MatchDay',
                                              'TeamHomeForm5', 'TeamAwayForm5',
                                              'OpponentForm5',
                                              'OpponentWin...
                                                               learning_rate=0.005,
                                                            